# Flippy Illusions: experimental SDXL + SD1

Optimize one shared Fourier image for an upright prompt and a prompt seen after a 180° rotation. This uses score distillation, not ordinary text-to-image inference. SDXL is experimental: execution, illusion quality, speed, and memory are separate questions.

**Start here:** select **Runtime → Change runtime type → GPU**, then run cells 1–7 in order. The default is a **four-step, 512-pixel exploratory smoke test**, including save/resume. Both views must receive gradients. For a longer SDXL reference experiment, select `MODE = "experiment"` in cell 5: its model input is 1024 square and the learnable image is 256 square.

No GPU generation or quality comparison has been validated by the author of this change yet. CPU contract tests and tiny random-component integration checks do not establish real-model compatibility. GPU assignment and available VRAM vary in Colab. No paid hosting is needed; free Colab resources are not guaranteed.


## 1. Select the fork and source revision

Run this cell again after restarting the runtime. It preserves local edits and refuses to overwrite another repository. A dedicated sparse checkout avoids downloading unrelated images and UV maps. Set `REF` to a branch, tag, or full commit SHA.


In [ ]:
# Safe setup: use a dedicated checkout of your fork. No files are deleted.
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/adrianIam/Diffusion-Illusions.git"
REF = "codex/sdxl-flippy-colab"  # Change to master after this branch is merged, or use a commit SHA.
CHECKOUT = Path("/content/Diffusion-Illusions")
def git(*args):
    return subprocess.check_output(["git", "-C", str(CHECKOUT), *args], text=True).strip()

if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    REPO_URL, str(CHECKOUT)], check=True)
elif not (CHECKOUT / ".git").is_dir():
    raise RuntimeError(f"{CHECKOUT} exists and is not a Git checkout. Choose another CHECKOUT path.")
if git("remote", "get-url", "origin").removesuffix(".git") != REPO_URL.removesuffix(".git"):
    raise RuntimeError("Checkout belongs to another repository. Choose another CHECKOUT path.")
if git("status", "--porcelain"):
    raise RuntimeError("Checkout has local edits. Preserve them before selecting a ref, or use another path.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", REF], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "sparse-checkout", "set", "source", "configs", "scripts", "docs", "tests"], check=True)
os.chdir(CHECKOUT)
print("Source commit:", git("rev-parse", "HEAD"))


## 2. Install the separate SDXL dependency set

Run once per fresh runtime. **After installation, use Runtime → Restart session**, then rerun cell 1 and continue at cell 3. Keeping Colab's CUDA-enabled PyTorch avoids replacing it with a CPU build. Tested locally: Python 3.12, PyTorch 2.6.0+cpu, Diffusers 0.35.1, Transformers 4.56.2. Your Colab CUDA build is recorded in the manifest; its real-model compatibility still needs the smoke test.

If imports fail after restarting, keep the traceback and package versions for diagnosis. The historical notebooks have their own `requirements.txt`; do not install that file into this runtime.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-sdxl.txt"], check=True)
print("Installation finished. Restart the session, rerun cell 1, then continue at cell 3.")


## 3. Check the GPU and imports

Stop here if no GPU is attached. There is no lengthy CPU generation fallback. The public SDXL base model normally downloads without a login. If the Hub actually returns an authentication or license error (more likely for an SD1 model), follow that model's access instructions and call `huggingface_hub.notebook_login()` interactively. Never paste tokens into this notebook or share them in logs.


In [ ]:
import importlib.metadata
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline
from source.flippy import FlippyConfig, run

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Select Runtime > Change runtime type > GPU and reconnect.")
props = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print(f"GPU: {props.name} | total {total / 2**30:.1f} GiB | free {free / 2**30:.1f} GiB")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
for package in ["diffusers", "transformers", "accelerate", "huggingface-hub"]:
    print(package, importlib.metadata.version(package))


## 4. Choose persistent storage (optional)

Colab's local files disappear when its runtime is recycled. Select Google Drive before a long run if you need persistence; checkpoint files can be large. Atomic rename protects ordinary interrupted writes, but a disconnected Drive mount can still lose unsynced data. Keep your own downloaded copy too.


In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/DiffusionIllusions")
else:
    OUTPUT_ROOT = Path("/content/flippy-results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)


## 5. Configure the experiment or resume

Start with `MODE = "smoke"`. The smoke preset intentionally uses 512-pixel SDXL input, a 64-pixel learnable image and a smaller MLP, to test execution before attempting 1024. Lower resolution may reduce quality. SDXL guidance starts at 20; the SD1 reference uses the historical 60. These are starting settings, not tuned quality claims.

`iterations` is the total across resumes. `max_seconds` is an optional per-session optimization budget, excluding model loading. A random iteration scores one view (as upstream does); `both` scores both views and costs two UNet evaluations. Weights are normalized to sum to two, as in the original notebook.

To resume, set `RESUME_CHECKPOINT` to **your own trusted** `checkpoint.pt`, set `MODE = "experiment"`, and raise `TARGET_ITERATIONS` if desired. Stored model, prompts, image sizes, dtype, seed, optimizer settings and device/dependency identity must match. Only output directory, save interval and budgets may change. Do not load someone else's checkpoint: optimizer/RNG restoration uses Python pickle.


In [ ]:
from dataclasses import replace
from datetime import datetime, timezone
import json

MODE = "smoke"  # "smoke" or "experiment"
BACKEND = "sdxl"  # "sdxl" or "sd1"
SEED = 0
TARGET_ITERATIONS = 1000
MAX_SECONDS = None  # Example: 1800 for a bounded 30-minute optimization session.
RESUME_CHECKPOINT = ""  # Example: /content/drive/MyDrive/DiffusionIllusions/.../checkpoint.pt

name = f"{BACKEND}-{MODE}-seed{SEED}-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
if RESUME_CHECKPOINT:
    saved = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=False)
    config = FlippyConfig(**saved["config"])
    config = replace(config, iterations=max(TARGET_ITERATIONS, saved["iteration"]),
                     max_seconds=MAX_SECONDS, output_dir=str(Path(RESUME_CHECKPOINT).parent))
elif MODE == "smoke":
    config = FlippyConfig.smoke(BACKEND, seed=SEED, output_dir=str(OUTPUT_ROOT / name))
elif MODE == "experiment":
    config = FlippyConfig(backend=BACKEND, seed=SEED, iterations=TARGET_ITERATIONS,
                         guidance_scale=20. if BACKEND == "sdxl" else 60.,
                         max_seconds=MAX_SECONDS, output_dir=str(OUTPUT_ROOT / name))
else:
    raise ValueError("MODE must be smoke or experiment")

# Edit these before a NEW run. Defaults are the original giraffe/penguin pair.
# config.prompt_a = "an intricate detailed hb pencil sketch of a giraffe head"
# config.prompt_b = "an intricate detailed hb pencil sketch of a penguin"
# config.negative_prompt = ""
# config.learning_rate = 1e-4
# config.guidance_scale = 20.  # SDXL may require retuning from SD1's high guidance.
# config.view_weights = (1., 1.)
# config.min_timestep, config.max_timestep = .02, .98
# config.dtype, config.vae_dtype = "auto", "float32"
# config.model_size = 512  # Lower-memory exploration, not the 1024 SDXL reference.
# config.attention_slicing = True  # Explicit alternative to default SDPA; can be slower.
print(json.dumps(config.__dict__, indent=2))


## 6. Run, preview both views, and verify save/resume

The default smoke test runs two steps, saves the model/optimizer/RNG state, reloads the real model and resumes to four steps. It checks finite nonzero image gradients, changed image parameters, frozen model parameters, both view counts, conditioning shapes and output writes. A successful smoke test establishes execution only. GPU kernels may be nondeterministic; this does not assert bitwise GPU equivalence.

On OOM, the last complete checkpoint is preserved and the manifest records the failure. For a **new** lower-memory run, use `image_size=64`, `hidden_dim=64`, `num_features=32`, `model_size=512`, and optionally `attention_slicing=True`. Do not silently change a resumed experiment. If that still fails, capture the traceback and GPU summary. Do not keep retrying an oversized configuration.


In [ ]:
from IPython.display import display, clear_output

def show_progress(iteration, preview, metrics):
    clear_output(wait=True)
    print(f"Iteration {iteration}/{config.iterations}; optimization {metrics['elapsed_seconds']:.1f}s")
    print("Scored views (upright, flipped):", metrics["view_steps"])
    print("Gradient norm:", metrics["last_gradient_norm"])
    print(f"Peak allocated {metrics['peak_allocated_bytes'] / 2**30:.2f} GiB; "
          f"reserved {metrics['peak_reserved_bytes'] / 2**30:.2f} GiB")
    display(preview)

if MODE == "smoke" and not RESUME_CHECKPOINT:
    result = run(replace(config, iterations=2), callback=show_progress)
    result = run(config, resume=result["checkpoint"], callback=show_progress)
else:
    result = run(config, resume=RESUME_CHECKPOINT or None, callback=show_progress)

print(json.dumps(result, indent=2))
manifest = json.loads((Path(result["output_dir"]) / "manifest.json").read_text())
if MODE == "smoke":
    assert result["iteration"] == 4
    assert min(result["metrics"]["view_steps"]) >= 4
    assert result["metrics"]["image_parameters_changed"]
    assert result["metrics"]["frozen_model_gradients_absent"]
    assert result["metrics"]["last_gradient_norm"] > 0
    assert len(result["metrics"]["sessions"]) == 2
    assert all((Path(result["output_dir"]) / f).exists() for f in
               ["checkpoint.pt", "manifest.json", "image.png", "upright.png", "flipped.png", "comparison.png"])
    print("GPU smoke + save/resume passed. Visual quality is still unassessed.")
print("Conditioning shapes:", manifest["backend"]["conditioning_shapes"])


## 7. Inspect and download results

`image.png` / `upright.png`, `flipped.png`, `comparison.png`, periodic previews, `checkpoint.pt`, and `manifest.json` are saved in the output directory. The manifest includes prompts, seed, source commit, model revision when obtainable, dependency versions, resolutions, view counts, timing and peak GPU memory.

Set `DOWNLOAD = True` to download a ZIP. It includes the checkpoint and may be large. To diagnose a failure, return the manifest, traceback, GPU summary, package versions and final comparison PNG; only share your checkpoint if needed and trusted. If a run fails during initial model download before a manifest is created, return the setup/GPU cell output and traceback.


In [ ]:
from PIL import Image
folder = Path(result["output_dir"])
display(Image.open(folder / "comparison.png"))
print("Files saved to:", folder)
DOWNLOAD = False
if DOWNLOAD:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(folder), "zip", root_dir=folder)
    files.download(archive)


## Longer runs and SD1 comparison

After the smoke test passes, return to cell 5, select `MODE = "experiment"`, choose a fresh output directory and run cells 5–7. Start with 1,000 iterations; extend the saved run only after inspecting both views. The 1024 SDXL setting may exceed your assigned GPU's VRAM.

For a reproducible comparison, use three prompt pairs × two seeds × two backends. `python scripts/compare_flippy.py --output-root /content/flippy-comparison` writes twelve explicit configurations; it does not start them. Add `--run` only when you want that bounded batch to execute. See `docs/sdxl-flippy.md` for equal-iteration/equal-time commands and a human scoring rubric. Do not interpret matched seeds as matched model noise trajectories.

The selectable SD1 backend retains the historical noise-only SDS weighting, Fourier representation, random one-view updates, SGD learning rate, and the 512-pixel / 0.18215 defaults **through its model config**. Modern dependencies, frozen model weights and improved output/checkpoint handling are recorded differences. The historical `flippy_illusions_for_colab.ipynb` remains available with safe fork-aware setup. Its unpinned historical environment has not been reconstructed, so an exact historical quality/performance baseline remains pending.
